# Amazon ML Challenge — Business Entity Resolution
## Full-scale, Kaggle-compatible pipeline

This notebook implements the candidate-generation + feature + matching-model
pipeline for the Business Entity Resolution challenge, designed to run on a
Kaggle CPU notebook with an approximately **20 GB working-disk** budget and
**millions of records** (full TRAIN: S1=2,206,821 / S2=5,034,616 / S3=5,285,603).

It is a scaled-up, chunked/streaming rewrite of a reference implementation
that was measured (on a 10,000 S1 / 100,000 S2 / 100,000 S3 ID-aligned dev
subset) at:

```
S2 candidate recall : 98.26%  (16,439 / 16,730)
S3 candidate recall : 97.68%  (17,603 / 18,022)
Combined recall      : 97.96%  (34,042 / 34,752)
Candidate volume     : 2,541,867 pairs  (254.19 candidates/S1)
```

That reference run took 1,565.7s on 10K S1 rows using a pure-Python
Levenshtein fallback and a single monolithic output file — **neither of
those two things will work at full scale**, so this notebook:

- uses `rapidfuzz` (C-accelerated, MIT license) for edit-distance features
  instead of pure Python
- never materializes one giant candidate/feature file — everything is
  processed in **S1 chunks** and written as **Parquet shards**
- monitors RAM/disk at every stage and reports it
- keeps the **same 17 blocking-key strategy** that produced the 97.96%
  reference number (no blocking rule silently removed)

**Honesty note:** the 97.96% number above is the verified *reference*
result from the smaller dev subset. This notebook has not yet been run
end-to-end against the full 2.2M/5M/5.3M data in this environment — do
that by running the cells in Kaggle. Anywhere this notebook can't verify
a number without Kaggle's environment/data, it says so instead of
guessing.


## 1. Install / import dependencies

In [1]:
# rapidfuzz: C-accelerated Levenshtein (MIT license) -- fixes the #1 measured
# bottleneck from the reference run (pure-Python Levenshtein, ~616us/pair).
# lightgbm: MIT license, <8B params (it's a GBDT, not an LLM) -- satisfies the
# competition's model-license/size constraint.
import sys, subprocess

def pip_install(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

for pkg in ["rapidfuzz", "lightgbm", "pyarrow", "psutil"]:
    try:
        __import__(pkg if pkg != "lightgbm" else "lightgbm")
    except ImportError:
        pip_install(pkg)

import os
import gc
import csv
import json
import time
import shutil
import sqlite3
import unicodedata
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import psutil

try:
    from rapidfuzz.distance.Levenshtein import normalized_similarity as _lev_sim
    HAVE_RAPIDFUZZ = True
except ImportError:
    HAVE_RAPIDFUZZ = False

import lightgbm as lgb

print("rapidfuzz available:", HAVE_RAPIDFUZZ)
print("lightgbm version:", lgb.__version__)


ERROR: Could not find a version that satisfies the requirement rapidfuzz (from versions: none)
ERROR: No matching distribution found for rapidfuzz


rapidfuzz available: False
lightgbm version: 4.6.0


## 2. Locate Kaggle input files

Kaggle mounts input datasets read-only under `/kaggle/input/<dataset-name>/...`.
We don't hardcode the dataset slug (it depends on what you name the uploaded
dataset) -- instead we search for the four expected filenames anywhere under
`/kaggle/input`. If you're running this notebook outside Kaggle (e.g. to
smoke-test on a small local sample), set `DATA_DIR` manually instead.


In [2]:
def find_train_files(search_root="/kaggle/input"):
    """
    Search recursively for the four expected TRAIN files and return their
    paths. Falls back to None for any file not found -- callers must check.
    """
    wanted = {
        "train_source1.tsv": None,
        "train_source2.tsv": None,
        "train_source3.tsv": None,
        "train_ground_truth.tsv": None,
    }

    root = Path(search_root)
    if root.exists():
        for p in root.rglob("*.tsv"):
            if p.name in wanted and wanted[p.name] is None:
                wanted[p.name] = str(p)

    return wanted


DATA_DIR = None  # e.g. "/kaggle/working/local_sample" to override auto-discovery

if DATA_DIR is None:
    found = find_train_files()
    print(json.dumps(found, indent=2))
    missing = [k for k, v in found.items() if v is None]
    if missing:
        print(f"WARNING: could not locate {missing} under /kaggle/input. "
              f"Set DATA_DIR manually or check the dataset is attached.")
    S1_PATH = found["train_source1.tsv"]
    S2_PATH = found["train_source2.tsv"]
    S3_PATH = found["train_source3.tsv"]
    GT_PATH = found["train_ground_truth.tsv"]
else:
    S1_PATH = str(Path(DATA_DIR) / "train_source1.tsv")
    S2_PATH = str(Path(DATA_DIR) / "train_source2.tsv")
    S3_PATH = str(Path(DATA_DIR) / "train_source3.tsv")
    GT_PATH = str(Path(DATA_DIR) / "train_ground_truth.tsv")

WORK_DIR = Path("/kaggle/working")
OUT_DIR = WORK_DIR / "output"
SHARD_DIR = WORK_DIR / "feature_shards"
IDX_DIR = WORK_DIR / "index"
for d in (OUT_DIR, SHARD_DIR, IDX_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("S1_PATH:", S1_PATH)
print("S2_PATH:", S2_PATH)
print("S3_PATH:", S3_PATH)
print("GT_PATH:", GT_PATH)


{
  "train_source1.tsv": "/kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_source1.tsv",
  "train_source2.tsv": "/kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_source2.tsv",
  "train_source3.tsv": "/kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_source3.tsv",
  "train_ground_truth.tsv": "/kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_ground_truth.tsv"
}
S1_PATH: /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_source1.tsv
S2_PATH: /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_source2.tsv
S3_PATH: /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/st

## 3. Resource-monitoring utilities

Called after every major stage (per Part 16/20 of the task spec) so we never
"wait for Kaggle to crash" -- we print RAM/disk after each stage and can
bail out early if a budget is at risk.


In [3]:
def check_memory():
    vm = psutil.virtual_memory()
    proc = psutil.Process(os.getpid())
    return {
        "system_used_gb": round(vm.used / 1e9, 2),
        "system_available_gb": round(vm.available / 1e9, 2),
        "system_percent": vm.percent,
        "process_rss_gb": round(proc.memory_info().rss / 1e9, 2),
    }


def check_disk_space(path="/kaggle/working"):
    total, used, free = shutil.disk_usage(path)
    return {
        "total_gb": round(total / 1e9, 2),
        "used_gb": round(used / 1e9, 2),
        "free_gb": round(free / 1e9, 2),
    }


def cleanup_temp_files(paths):
    for p in paths:
        p = Path(p)
        try:
            if p.is_file():
                p.unlink()
            elif p.is_dir():
                shutil.rmtree(p)
        except FileNotFoundError:
            pass
    gc.collect()


def resource_report(stage_name, **extra):
    mem = check_memory()
    disk = check_disk_space()
    print("=" * 70)
    print(f"STAGE: {stage_name}")
    for k, v in extra.items():
        print(f"  {k}: {v}")
    print(f"  RAM  -> process={mem['process_rss_gb']}GB  "
          f"system_used={mem['system_used_gb']}GB  "
          f"system_available={mem['system_available_gb']}GB")
    print(f"  DISK -> used={disk['used_gb']}GB  free={disk['free_gb']}GB")
    print("=" * 70)
    return {"stage": stage_name, "memory": mem, "disk": disk, **extra}


resource_report("startup")


STAGE: startup
  RAM  -> process=0.3GB  system_used=0.98GB  system_available=32.2GB
  DISK -> used=16.01GB  free=4.93GB


{'stage': 'startup',
 'memory': {'system_used_gb': 0.98,
  'system_available_gb': 32.2,
  'system_percent': 4.3,
  'process_rss_gb': 0.3},
 'disk': {'total_gb': 20.96, 'used_gb': 16.01, 'free_gb': 4.93}}

## 4. Configuration

Country is treated as an open string set (per the problem statement: TEST
adds France, which never appears in TRAIN) -- nothing below hardcodes
`{US, India}`.


In [4]:
CONFIG = {
    # Per-key-type max bucket size. If a key's bucket (in the index actually
    # built) exceeds this, it is skipped for that record -- this is what
    # keeps candidate volume bounded without deleting any blocking rule.
    # Same values as the validated reference run.
    "freq_caps": {
        "NP6": 300, "NP8": 300, "NS6": 300,
        "FL": 400, "FL3": 400, "F2": 400, "FML": 400,
        "AN": 400, "A2": 400, "AW2": 400, "AL2": 80, "NL": 400,
        "A4W": 10, "AT2": 20, "RAT20": 20,
        "UNP4": 300, "UNP6": 300, "UFL": 300,
    },
    "s1_chunk_size": 20_000,       # S1 rows processed per chunk
    "index_backend": "sqlite",      # "memory" or "sqlite" -- sqlite recommended at full scale
    "a4_window": 3,
}

print(json.dumps(CONFIG, indent=2))


{
  "freq_caps": {
    "NP6": 300,
    "NP8": 300,
    "NS6": 300,
    "FL": 400,
    "FL3": 400,
    "F2": 400,
    "FML": 400,
    "AN": 400,
    "A2": 400,
    "AW2": 400,
    "AL2": 80,
    "NL": 400,
    "A4W": 10,
    "AT2": 20,
    "RAT20": 20,
    "UNP4": 300,
    "UNP6": 300,
    "UFL": 300
  },
  "s1_chunk_size": 20000,
  "index_backend": "sqlite",
  "a4_window": 3
}


## 5. Normalization

Unchanged logic from the validated reference implementation (name/address
normalization + legal-suffix stripping + address-abbreviation folding).


In [5]:
import re

def normalize_text(text):
    if not text:
        return ""
    text = str(text).lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


_NAME_SUFFIXES = [
    "incorporated", "inc", "corporation", "corp", "limited", "ltd",
    "llc", "company", "co", "private limited", "pvt ltd",
]

def normalize_business_name(name):
    name = normalize_text(name)
    words = name.split()
    while words and " ".join(words[-2:]) in _NAME_SUFFIXES:
        words = words[:-2]
    while words and words[-1] in _NAME_SUFFIXES:
        words = words[:-1]
    return " ".join(words)


_ADDR_REPLACEMENTS = {
    " street ": " st ", " road ": " rd ", " avenue ": " ave ",
    " boulevard ": " blvd ", " drive ": " dr ", " lane ": " ln ",
    " highway ": " hwy ", " apartment ": " apt ",
}

def normalize_address(address):
    address = normalize_text(address)
    address = " " + address + " "
    for old, new in _ADDR_REPLACEMENTS.items():
        address = address.replace(old, new)
    return address.strip()


# quick self-check (does not touch the real data)
assert normalize_business_name("Payne Enterprises LLC") == normalize_business_name("Payne Enterprises")
assert normalize_address("3315 Fremont Street, Peoria, IL") == normalize_address("3315 FREMONT ST, PEORIA, IL")
print("normalization self-check OK")


normalization self-check OK


## 6. Blocking strategies (all 17 key types from the validated reference)

Name: `NP6 NP8 NS6 FL FL3 F2 FML` (ASCII) + `UNP4 UNP6 UFL` (Unicode-preserving,
critical since 15.2%/11.5% of S2/S3 names are non-ASCII).
Address: `AN A2 AW2 AL2 NL` (original) + `A4W` (windowed number+nearby-word,
O(n) not O(n^2)) + `AT2` (unordered informative-token-pair) + `RAT20`
(rare-address-token, frequency-capped).

Country is folded into every key at indexing time (see Section 7's
`compose_key`) so blocking stays country-aware without special-casing it here.


In [6]:
def make_name_keys(name):
    n = normalize_business_name(name)
    if not n:
        return []
    keys = []
    compact = n.replace(" ", "")
    if len(compact) >= 6:
        keys.append(("NP6", compact[:6]))
    if len(compact) >= 8:
        keys.append(("NP8", compact[:8]))
    if len(compact) >= 6:
        keys.append(("NS6", compact[-6:]))
    words = n.split()
    if len(words) >= 2:
        first, last = words[0], words[-1]
        if len(first) >= 3 and len(last) >= 3:
            keys.append(("FL", first[:4] + "|" + last[:4]))
            keys.append(("FL3", first[:3] + "|" + last[:5]))
        w1, w2 = words[0], words[1]
        if len(w1) >= 3 and len(w2) >= 3:
            keys.append(("F2", w1[:5] + "|" + w2[:5]))
    if len(words) >= 3:
        first, middle, last = words[0], words[-2], words[-1]
        if len(first) >= 3 and len(last) >= 3:
            keys.append(("FML", first[:3] + "|" + middle[:3] + "|" + last[:3]))
    return keys


def make_unicode_name_keys(name):
    if not name:
        return []
    n = unicodedata.normalize("NFKC", str(name)).casefold()
    n = " ".join(n.split())
    if not n:
        return []
    keys = []
    compact = "".join(ch for ch in n if not ch.isspace())
    if len(compact) >= 4:
        keys.append(("UNP4", compact[:4]))
    if len(compact) >= 6:
        keys.append(("UNP6", compact[:6]))
    words = n.split()
    if len(words) >= 2:
        first, last = words[0], words[-1]
        if len(first) >= 2 and len(last) >= 2:
            keys.append(("UFL", first[:3] + "|" + last[:3]))
    return keys


def make_address_keys(address):
    a = normalize_address(address)
    if not a:
        return []
    keys = []
    words = a.split()
    if not words:
        return []
    numbers = [w for w in words if w.isdigit()]
    non_numeric = [w for w in words if not w.isdigit()]
    if numbers:
        number = numbers[0]
        for word in words:
            if not word.isdigit() and len(word) >= 3:
                keys.append(("AN", number + "|" + word[:6]))
                break
        if len(non_numeric) >= 2:
            keys.append(("A2", number + "|" + non_numeric[0][:4] + "|" + non_numeric[1][:4]))
    if len(non_numeric) >= 2:
        w1, w2 = non_numeric[0], non_numeric[1]
        if len(w1) >= 3 and len(w2) >= 3:
            keys.append(("AW2", w1[:5] + "|" + w2[:5]))
        w1, w2 = non_numeric[-2], non_numeric[-1]
        if len(w1) >= 3 and len(w2) >= 3:
            keys.append(("AL2", w1[:5] + "|" + w2[:5]))
    if numbers and non_numeric:
        number = numbers[0]
        last_word = non_numeric[-1]
        if len(last_word) >= 3:
            keys.append(("NL", number + "|" + last_word[:6]))
    return keys


GENERIC_ADDRESS_WORDS = {
    "street", "st", "road", "rd", "avenue", "ave", "lane", "ln", "drive", "dr",
    "boulevard", "blvd", "highway", "hwy", "near", "opposite", "opp",
    "building", "bldg", "floor", "fl", "block", "district", "city", "state",
    "india", "private", "limited", "ltd", "pvt",
}

def make_rare_address_token_keys(address):
    a = normalize_address(address)
    if not a:
        return []
    keys, seen = [], set()
    for tok in a.split():
        if tok.isdigit() or len(tok) < 4 or tok in GENERIC_ADDRESS_WORDS:
            continue
        token = tok[:12]
        if token and token not in seen:
            seen.add(token)
            keys.append(("RAT20", token))
    return keys


def make_at2_keys(address):
    a = normalize_address(address)
    if not a:
        return []
    informative = [t for t in a.split()
                   if not t.isdigit() and len(t) >= 4 and t not in GENERIC_ADDRESS_WORDS]
    if len(informative) < 2:
        return []
    informative = sorted(informative, key=lambda x: (-len(x), x))[:6]
    keys, seen = [], set()
    for i in range(len(informative) - 1):
        w1, w2 = informative[i][:8], informative[i + 1][:8]
        if w1 == w2:
            continue
        a1, a2 = sorted((w1, w2))
        key_value = a1 + "|" + a2
        if key_value not in seen:
            seen.add(key_value)
            keys.append(("AT2", key_value))
    return keys


def make_a4_keys(address, window=None):
    window = window or CONFIG["a4_window"]
    a = normalize_address(address)
    if not a:
        return []
    tokens = a.split()
    number_positions = [i for i, t in enumerate(tokens) if t.isdigit()]
    if not number_positions:
        return []
    keys, seen = [], set()
    for pos in number_positions:
        number = tokens[pos]
        nearby = []
        for offset in range(1, window + 1):
            for idx in (pos - offset, pos + offset):
                if 0 <= idx < len(tokens):
                    tok = tokens[idx]
                    if not tok.isdigit() and len(tok) >= 2:
                        nearby.append(tok)
        nearby = list(dict.fromkeys(nearby))[:3]
        for i in range(len(nearby)):
            for j in range(i + 1, len(nearby)):
                w1, w2 = sorted((nearby[i][:6], nearby[j][:6]))
                key_value = number + "|" + w1 + "|" + w2
                if key_value not in seen:
                    seen.add(key_value)
                    keys.append(("A4W", key_value))
    return keys


def make_all_keys(name, address):
    return (
        make_name_keys(name)
        + make_unicode_name_keys(name)
        + make_address_keys(address)
        + make_rare_address_token_keys(address)
        + make_a4_keys(address)
        + make_at2_keys(address)
    )

print("blocking key generators ready:",
      sorted({k for k, _ in make_all_keys("Test Business Pvt Ltd", "123 Main Street, Test City")}))


blocking key generators ready: ['A2', 'A4W', 'AL2', 'AN', 'AT2', 'F2', 'FL', 'FL3', 'NL', 'NP6', 'NP8', 'NS6', 'RAT20', 'UFL', 'UNP4', 'UNP6']


## 7. Build scalable S2/S3 indexes

Two backends:

- **`MemoryIndex`**: in-RAM dict-based posting lists. Fine for dev-scale
  subsets; risky for the full 5M+ record S2/S3 at once.
- **`SqliteIndex`**: disk-backed posting lists in a SQLite file with an
  index on `(key_type, key_value)`. This is the backend actually used for
  the full run (`CONFIG["index_backend"] = "sqlite"`), because it keeps
  RAM bounded regardless of dataset size -- the tradeoff being query
  latency, which is paid once per lookup during candidate generation.

Entity IDs are stored compactly (source prefix stripped, numeric suffix
kept as int/str) to reduce per-record overhead at 5M+ scale.


In [7]:
def split_entity_id(entity_id):
    if "-" not in entity_id:
        return "", entity_id
    prefix, suffix = entity_id.split("-", 1)
    return (prefix, int(suffix)) if suffix.isdigit() else (prefix, suffix)


def compose_key(country, key_value):
    return (country or "").strip().upper() + "||" + key_value


class MemoryIndex:
    def __init__(self, source_prefix):
        self.source_prefix = source_prefix
        self.postings = defaultdict(lambda: defaultdict(list))
        self.attrs = {}
        self.count = 0

    def add_record(self, entity_id, name, address, country):
        _, suffix = split_entity_id(entity_id)
        self.attrs[suffix] = (name, address, country)
        for key_type, key_value in make_all_keys(name, address):
            self.postings[key_type][compose_key(country, key_value)].append(suffix)
        self.count += 1

    def bucket(self, key_type, key_value):
        return self.postings.get(key_type, {}).get(key_value, [])

    def get_attrs(self, suffix):
        return self.attrs.get(suffix)

    def num_keys(self):
        return sum(len(v) for v in self.postings.values())


class SqliteIndex:
    def __init__(self, source_prefix, db_path):
        self.source_prefix = source_prefix
        self.db_path = db_path
        self.count = 0
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("PRAGMA synchronous = OFF")
        self.conn.execute("PRAGMA journal_mode = MEMORY")
        self.conn.execute("CREATE TABLE IF NOT EXISTS postings (key_type TEXT, key_value TEXT, suffix TEXT)")
        self.conn.execute("CREATE TABLE IF NOT EXISTS attrs (suffix TEXT PRIMARY KEY, name TEXT, address TEXT, country TEXT)")
        self._buf, self._abuf = [], []

    def add_record(self, entity_id, name, address, country):
        _, suffix = split_entity_id(entity_id)
        suffix = str(suffix)
        self._abuf.append((suffix, name, address, country))
        for key_type, key_value in make_all_keys(name, address):
            self._buf.append((key_type, compose_key(country, key_value), suffix))
        self.count += 1
        if len(self._buf) >= 300_000:
            self._flush()

    def _flush(self):
        if self._buf:
            self.conn.executemany("INSERT INTO postings VALUES (?,?,?)", self._buf)
            self._buf = []
        if self._abuf:
            self.conn.executemany("INSERT OR REPLACE INTO attrs VALUES (?,?,?,?)", self._abuf)
            self._abuf = []

    def finalize(self):
        self._flush()
        self.conn.execute("CREATE INDEX IF NOT EXISTS idx_postings ON postings(key_type, key_value)")
        self.conn.commit()

    def bucket(self, key_type, key_value):
        cur = self.conn.execute(
            "SELECT suffix FROM postings WHERE key_type=? AND key_value=?", (key_type, key_value)
        )
        return [row[0] for row in cur.fetchall()]

    def get_attrs(self, suffix):
        cur = self.conn.execute(
            "SELECT name, address, country FROM attrs WHERE suffix=?", (str(suffix),)
        )
        return cur.fetchone()

    def num_keys(self):
        cur = self.conn.execute("SELECT COUNT(*) FROM (SELECT DISTINCT key_type, key_value FROM postings)")
        return cur.fetchone()[0]


def read_tsv(path):
    with open(path, "r", encoding="utf-8", errors="replace", newline="") as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in reader:
            yield row


def build_index(path, source_prefix, backend="sqlite", db_path=None, limit=None):
    if backend == "sqlite":
        if db_path is None:
            db_path = str(IDX_DIR / f"{source_prefix}_index.sqlite")
        if Path(db_path).exists():
            Path(db_path).unlink()
        index = SqliteIndex(source_prefix, db_path)
    else:
        index = MemoryIndex(source_prefix)

    t0 = time.time()
    for i, row in enumerate(read_tsv(path)):
        if limit is not None and i >= limit:
            break
        index.add_record(
            entity_id=row.get("entity_id", ""),
            name=row.get("business_name", ""),
            address=row.get("business_address", ""),
            country=(row.get("country") or "").strip(),
        )
        if index.count % 1_000_000 == 0:
            print(f"  [{source_prefix}] indexed {index.count:,} records "
                  f"({time.time()-t0:.1f}s elapsed)")

    if backend == "sqlite":
        index.finalize()

    print(f"[{source_prefix}] index built: {index.count:,} records, "
          f"{index.num_keys():,} distinct keys, {time.time()-t0:.1f}s")
    return index


## 8. Generate candidates in chunks

`find_candidates` applies the per-key-type frequency cap from `CONFIG`
against whatever index was actually built -- never a value hardcoded from
a smaller dev run. `iter_s1_chunks` streams S1 in fixed-size batches so we
never hold the full 2.2M-row S1 file, or all of its candidates, in memory
at once.


In [8]:
def find_candidates(row, index, freq_caps):
    country = (row.get("country") or "").strip()
    name = row.get("business_name", "")
    address = row.get("business_address", "")

    candidates = {}
    for key_type, key_value in make_all_keys(name, address):
        cap = freq_caps.get(key_type, 500)
        composed = compose_key(country, key_value)
        bucket = index.bucket(key_type, composed)
        bucket_size = len(bucket)
        if bucket_size == 0 or bucket_size > cap:
            continue
        for suffix in bucket:
            entry = candidates.get(suffix)
            if entry is None:
                candidates[suffix] = {"key_types": {key_type}, "min_freq": bucket_size}
            else:
                entry["key_types"].add(key_type)
                entry["min_freq"] = min(entry["min_freq"], bucket_size)
    return candidates


def generate_candidate_pairs_for_row(s1_row, indexes_by_source, freq_caps):
    s1_id = s1_row.get("entity_id", "")
    rows = []
    for source, index in indexes_by_source.items():
        for suffix, info in find_candidates(s1_row, index, freq_caps).items():
            rows.append({
                "source1_entity_id": s1_id,
                "candidate_entity_id": f"{source}-{suffix}",
                "source": source,
                "blocking_key_types": "|".join(sorted(info["key_types"])),
                "blocking_num_keys": len(info["key_types"]),
                "blocking_min_freq": info["min_freq"],
            })
    return rows


def iter_s1_chunks(path, chunk_size, limit=None):
    chunk = []
    n = 0
    for row in read_tsv(path):
        if limit is not None and n >= limit:
            break
        chunk.append(row)
        n += 1
        if len(chunk) >= chunk_size:
            yield chunk
            chunk = []
    if chunk:
        yield chunk


## 9. Candidate recall evaluation

Reports S2 recall, S3 recall and combined recall separately (never treating
mixed S2+S3 ground truth as S2-only recall), matching the metric definition
used to validate the 97.96% reference number.


In [9]:
def load_ground_truth(path):
    ground_truth = {}
    for row in read_tsv(path):
        s1_id = row["source1_entity_id"]
        raw = (row.get("matched_entity_ids") or "").strip()
        all_ids = {x.strip() for x in raw.split(",") if x.strip()} if raw else set()
        ground_truth[s1_id] = {
            "S2": {x for x in all_ids if x.startswith("S2-")},
            "S3": {x for x in all_ids if x.startswith("S3-")},
            "all": all_ids,
        }
    return ground_truth


def label_for(s1_id, candidate_id, ground_truth):
    entry = ground_truth.get(s1_id)
    return int(entry is not None and candidate_id in entry["all"])


## 10. Feature generation (chunked, `rapidfuzz`-accelerated)

Same feature set as the validated reference (`features.py`), but the
Levenshtein similarity now uses `rapidfuzz` when available -- this directly
targets the measured ~616us/pair bottleneck from the reference run. Falls
back to a pure-Python implementation only if `rapidfuzz` genuinely isn't
installed, so the notebook still runs (slowly) even offline.


In [16]:
# ============================================================
# 10. FEATURE GENERATION
# Fast version — avoids pure-Python Levenshtein bottleneck
# ============================================================

from difflib import SequenceMatcher
import re


# ------------------------------------------------------------
# LEVENSHTEIN / TEXT SIMILARITY
# ------------------------------------------------------------

def _levenshtein_distance(a, b):
    """
    Original exact Levenshtein implementation.

    Kept here for reference/fallback compatibility, but
    levenshtein_similarity() below will NOT use it when
    RapidFuzz is unavailable because it is too slow at scale.
    """
    if a == b:
        return 0

    if len(a) < len(b):
        a, b = b, a

    if not b:
        return len(a)

    prev = list(range(len(b) + 1))

    for i, ca in enumerate(a, start=1):
        cur = [i] + [0] * len(b)

        for j, cb in enumerate(b, start=1):
            cost = 0 if ca == cb else 1

            cur[j] = min(
                cur[j - 1] + 1,
                prev[j] + 1,
                prev[j - 1] + cost
            )

        prev = cur

    return prev[-1]


def levenshtein_similarity(a, b):
    """
    Fast similarity function.

    If RapidFuzz is available:
        use true normalized Levenshtein similarity.

    If RapidFuzz is NOT available:
        use SequenceMatcher instead of the extremely slow
        pure-Python Levenshtein dynamic-programming algorithm.
    """

    if not a or not b:
        return 0.0

    if a == b:
        return 1.0

    # Preferred fast implementation
    if HAVE_RAPIDFUZZ:
        return _lev_sim(a, b)

    # Fast fallback
    return SequenceMatcher(None, a, b).ratio()


def text_similarity(a, b):
    """
    Character-level similarity using SequenceMatcher.
    """

    if not a or not b:
        return 0.0

    if a == b:
        return 1.0

    return SequenceMatcher(None, a, b).ratio()


# ------------------------------------------------------------
# TOKEN SIMILARITY
# ------------------------------------------------------------

def token_similarity(a, b):
    if not a or not b:
        return 0.0

    t1 = set(a.split())
    t2 = set(b.split())

    if not t1 or not t2:
        return 0.0

    return len(t1 & t2) / len(t1 | t2)


def token_overlap_count(a, b):
    if not a or not b:
        return 0

    return len(set(a.split()) & set(b.split()))


# ------------------------------------------------------------
# CHARACTER N-GRAM SIMILARITY
# ------------------------------------------------------------

def char_ngram_jaccard(a, b, n=2):
    if not a or not b:
        return 0.0

    if len(a) < n or len(b) < n:
        return 0.0

    g1 = {
        a[i:i+n]
        for i in range(len(a) - n + 1)
    }

    g2 = {
        b[i:i+n]
        for i in range(len(b) - n + 1)
    }

    if not g1 or not g2:
        return 0.0

    return len(g1 & g2) / len(g1 | g2)


# ------------------------------------------------------------
# PREFIX / SUFFIX
# ------------------------------------------------------------

def prefix_similarity(a, b, k=4):
    if not a or not b:
        return 0.0

    ap = a[:k]
    bp = b[:k]

    m = min(len(ap), len(bp))

    if not m:
        return 0.0

    return sum(
        1 for x, y in zip(ap, bp)
        if x == y
    ) / m


def suffix_similarity(a, b, k=4):
    if not a or not b:
        return 0.0

    as_ = a[-k:]
    bs = b[-k:]

    m = min(len(as_), len(bs))

    if not m:
        return 0.0

    return sum(
        1
        for x, y in zip(
            reversed(as_),
            reversed(bs)
        )
        if x == y
    ) / m


# ------------------------------------------------------------
# ADDRESS FEATURES
# ------------------------------------------------------------

def extract_address_numbers(address):
    a = normalize_address(address)

    return re.findall(
        r"\d+",
        a
    )


def address_number_match(addr1, addr2):
    n1 = extract_address_numbers(addr1)
    n2 = extract_address_numbers(addr2)

    if not n1 or not n2:
        return 0.5

    return 1.0 if n1 == n2 else 0.0


# ------------------------------------------------------------
# COUNTRY
# ------------------------------------------------------------

def country_match(c1, c2):
    if not c1 or not c2:
        return 0.0

    return (
        1.0
        if c1.strip().lower() == c2.strip().lower()
        else 0.0
    )


# ------------------------------------------------------------
# FEATURE COLUMN ORDER
# ------------------------------------------------------------

FEATURE_COLUMNS = [
    "name_exact_match",
    "name_char_similarity",
    "name_levenshtein_similarity",
    "name_token_jaccard",
    "name_token_overlap",
    "name_compact_similarity",
    "name_prefix_similarity",
    "name_suffix_similarity",
    "name_char_ngram_jaccard",

    "address_exact_match",
    "address_char_similarity",
    "address_levenshtein_similarity",
    "address_token_jaccard",
    "address_token_overlap",
    "address_number_match",
    "address_char_ngram_jaccard",

    "country_match",

    "name_missing",
    "address_missing",

    "combined_name_address_score",
]


# ------------------------------------------------------------
# MAIN FEATURE FUNCTION
# ------------------------------------------------------------

def calculate_features(
    name1,
    address1,
    country1,
    name2,
    address2,
    country2
):
    """
    Calculate all matching features for one S1/S2 or S1/S3 pair.
    """

    # Normalize once
    n1 = normalize_business_name(name1)
    n2 = normalize_business_name(name2)

    a1 = normalize_address(address1)
    a2 = normalize_address(address2)

    # --------------------------------------------------------
    # Name similarities
    # --------------------------------------------------------

    name_char = text_similarity(
        n1,
        n2
    )

    name_lev = levenshtein_similarity(
        n1,
        n2
    )

    name_token = token_similarity(
        n1,
        n2
    )

    name_overlap = token_overlap_count(
        n1,
        n2
    )

    name_compact = text_similarity(
        n1.replace(" ", ""),
        n2.replace(" ", "")
    )

    name_prefix = prefix_similarity(
        n1,
        n2
    )

    name_suffix = suffix_similarity(
        n1,
        n2
    )

    name_ngram = char_ngram_jaccard(
        n1,
        n2
    )

    # --------------------------------------------------------
    # Address similarities
    # --------------------------------------------------------

    address_char = text_similarity(
        a1,
        a2
    )

    address_lev = levenshtein_similarity(
        a1,
        a2
    )

    address_token = token_similarity(
        a1,
        a2
    )

    address_overlap = token_overlap_count(
        a1,
        a2
    )

    address_num = address_number_match(
        address1,
        address2
    )

    address_ngram = char_ngram_jaccard(
        a1,
        a2
    )

    # --------------------------------------------------------
    # Return feature dictionary
    # --------------------------------------------------------

    return {
        # Name
        "name_exact_match": int(
            bool(n1) and n1 == n2
        ),

        "name_char_similarity": name_char,

        "name_levenshtein_similarity": name_lev,

        "name_token_jaccard": name_token,

        "name_token_overlap": name_overlap,

        "name_compact_similarity": name_compact,

        "name_prefix_similarity": name_prefix,

        "name_suffix_similarity": name_suffix,

        "name_char_ngram_jaccard": name_ngram,

        # Address
        "address_exact_match": int(
            bool(a1) and a1 == a2
        ),

        "address_char_similarity": address_char,

        "address_levenshtein_similarity": address_lev,

        "address_token_jaccard": address_token,

        "address_token_overlap": address_overlap,

        "address_number_match": address_num,

        "address_char_ngram_jaccard": address_ngram,

        # Country
        "country_match": country_match(
            country1,
            country2
        ),

        # Missing values
        "name_missing": int(
            not n1 or not n2
        ),

        "address_missing": int(
            not a1 or not a2
        ),

        # Combined score
        "combined_name_address_score": (
            0.6 * name_char
            + 0.4 * address_char
        ),
    }


# ------------------------------------------------------------
# SELF TEST
# ------------------------------------------------------------

print("=" * 70)
print("FEATURE GENERATION CELL LOADED")
print("=" * 70)

print("RapidFuzz available:", HAVE_RAPIDFUZZ)
print("Number of features:", len(FEATURE_COLUMNS))

# Quick test
_test_features = calculate_features(
    "ABC Private Limited",
    "123 Main Street",
    "India",
    "ABC Pvt Ltd",
    "123 Main St",
    "India",
)

print("Feature test successful.")
print("Generated features:", len(_test_features))

assert set(_test_features.keys()) == set(FEATURE_COLUMNS)

print("Feature-column validation: OK")
print("=" * 70)

FEATURE GENERATION CELL LOADED
RapidFuzz available: False
Number of features: 20
Feature test successful.
Generated features: 20
Feature-column validation: OK


In [17]:
# ============================================================
# FAST 100-ROW BENCHMARK
# ============================================================

SMOKE_TEST_S1_LIMIT = 100

print("=" * 70)
print("STARTING FAST 100-ROW SMOKE TEST")
print("=" * 70)

t0 = time.time()

smoke_summary_fast = run_full_pipeline(
    S1_PATH,
    s2_index,
    s3_index,
    gt_path=GT_PATH,
    run_limit_s1=SMOKE_TEST_S1_LIMIT,
    tag="smoke_fast",
)

elapsed = time.time() - t0

print("\n" + "=" * 70)
print("FAST SMOKE TEST FINISHED")
print("=" * 70)

print(json.dumps(
    smoke_summary_fast,
    indent=2,
    default=str
))

print(f"\nTotal runtime: {elapsed:.2f} seconds")

resource_report(
    "after fast smoke test",
    elapsed_s=round(elapsed, 2),
)

STARTING FAST 100-ROW SMOKE TEST
STAGE: smoke_fast chunk 1
  s1_processed: 100
  candidate_pairs: 9869
  elapsed_s: 10.0
  RAM  -> process=3.52GB  system_used=4.27GB  system_available=28.91GB
  DISK -> used=8.68GB  free=12.26GB

FAST SMOKE TEST FINISHED
{
  "tag": "smoke_fast",
  "s1_processed": 100,
  "candidate_pairs": 9869,
  "avg_candidates_per_s1": 98.69,
  "elapsed_s": 10.012428760528564,
  "shards": 1,
  "positive_labels": 1,
  "negative_labels": 9868,
  "recall_by_source": {
    "S2": 0.006802721088435374,
    "S3": 0.0
  },
  "recall_combined": 0.003105590062111801
}

Total runtime: 46.01 seconds
STAGE: after fast smoke test
  elapsed_s: 46.01
  RAM  -> process=0.56GB  system_used=1.37GB  system_available=31.81GB
  DISK -> used=8.68GB  free=12.26GB


{'stage': 'after fast smoke test',
 'memory': {'system_used_gb': 1.37,
  'system_available_gb': 31.81,
  'system_percent': 5.5,
  'process_rss_gb': 0.56},
 'disk': {'total_gb': 20.96, 'used_gb': 8.68, 'free_gb': 12.26},
 'elapsed_s': 46.01}

In [18]:
# ============================================================
# 1,000-ROW BENCHMARK
# ============================================================

BENCHMARK_S1_LIMIT = 1000

print("=" * 70)
print("STARTING 1,000-ROW BENCHMARK")
print("=" * 70)

t0 = time.time()

benchmark_1000 = run_full_pipeline(
    S1_PATH,
    s2_index,
    s3_index,
    gt_path=GT_PATH,
    run_limit_s1=BENCHMARK_S1_LIMIT,
    tag="benchmark_1000",
)

elapsed = time.time() - t0

print("\n" + "=" * 70)
print("1,000-ROW BENCHMARK FINISHED")
print("=" * 70)

print(json.dumps(
    benchmark_1000,
    indent=2,
    default=str
))

print(f"\nTotal runtime: {elapsed:.2f} seconds")

resource_report(
    "after 1,000-row benchmark",
    elapsed_s=round(elapsed, 2),
)

STARTING 1,000-ROW BENCHMARK
STAGE: benchmark_1000 chunk 1
  s1_processed: 1000
  candidate_pairs: 100248
  elapsed_s: 66.2
  RAM  -> process=3.55GB  system_used=4.37GB  system_available=28.81GB
  DISK -> used=8.68GB  free=12.26GB

1,000-ROW BENCHMARK FINISHED
{
  "tag": "benchmark_1000",
  "s1_processed": 1000,
  "candidate_pairs": 100248,
  "avg_candidates_per_s1": 100.248,
  "elapsed_s": 66.2455484867096,
  "shards": 1,
  "positive_labels": 11,
  "negative_labels": 100237,
  "recall_by_source": {
    "S2": 0.0017584994138335288,
    "S3": 0.004417448923246825
  },
  "recall_combined": 0.0031276656241114587
}

Total runtime: 96.97 seconds
STAGE: after 1,000-row benchmark
  elapsed_s: 96.97
  RAM  -> process=0.6GB  system_used=1.39GB  system_available=31.79GB
  DISK -> used=8.68GB  free=12.26GB


{'stage': 'after 1,000-row benchmark',
 'memory': {'system_used_gb': 1.39,
  'system_available_gb': 31.79,
  'system_percent': 5.6,
  'process_rss_gb': 0.6},
 'disk': {'total_gb': 20.96, 'used_gb': 8.68, 'free_gb': 12.26},
 'elapsed_s': 96.97}

In [19]:
# ============================================================
# PROFILE FEATURE GENERATION
# ============================================================

import time

print("=" * 70)
print("PROFILING FEATURE CALCULATION")
print("=" * 70)

# Take ONE S1 row
sample_s1 = next(
    iter(iter_s1_chunks(
        S1_PATH,
        chunk_size=1,
        limit=1
    ))
)

s1_row = sample_s1[0]

# Generate candidates using our existing 20k/20k test indexes
t0 = time.time()

pairs = generate_candidate_pairs_for_row(
    s1_row,
    {
        "S2": s2_index,
        "S3": s3_index
    },
    CONFIG["freq_caps"]
)

candidate_time = time.time() - t0

print(f"Candidates generated: {len(pairs)}")
print(f"Candidate generation time: {candidate_time:.4f}s")


# ------------------------------------------------------------
# Time feature calculation only
# ------------------------------------------------------------

feature_times = []

for pair in pairs:

    source = pair["source"]
    candidate_id = pair["candidate_entity_id"]

    _, suffix = split_entity_id(candidate_id)

    attrs = {
        "S2": s2_index,
        "S3": s3_index
    }[source].get_attrs(suffix)

    if attrs:
        cand_name, cand_addr, cand_country = attrs
    else:
        cand_name, cand_addr, cand_country = "", "", ""

    t1 = time.time()

    feats = calculate_features(
        s1_row.get("business_name", ""),
        s1_row.get("business_address", ""),
        s1_row.get("country", ""),
        cand_name,
        cand_addr,
        cand_country,
    )

    feature_times.append(time.time() - t1)


total_feature_time = sum(feature_times)

print()
print("=" * 70)
print("FEATURE PROFILE")
print("=" * 70)

print(f"Candidates: {len(pairs)}")
print(f"Total feature time: {total_feature_time:.4f}s")
print(
    f"Average feature time/pair: "
    f"{total_feature_time / max(len(pairs), 1):.6f}s"
)

print(
    f"Estimated feature time for 100k pairs: "
    f"{total_feature_time / max(len(pairs), 1) * 100_000:.2f}s"
)

print(
    f"Estimated feature time for 1M pairs: "
    f"{total_feature_time / max(len(pairs), 1) * 1_000_000:.2f}s"
)

print("=" * 70)

PROFILING FEATURE CALCULATION
Candidates generated: 44
Candidate generation time: 0.0009s

FEATURE PROFILE
Candidates: 44
Total feature time: 0.0172s
Average feature time/pair: 0.000391s
Estimated feature time for 100k pairs: 39.14s
Estimated feature time for 1M pairs: 391.42s


## 11. Full-TRAIN chunked driver

This is the orchestration cell wiring sections 5-10 together, satisfying
Part 8's "never materialize all candidates" requirement:

```
S1 chunk -> candidates (S2 index + S3 index) -> features -> labels
  -> append to a Parquet shard -> discard chunk -> next chunk
```

Each shard is written and closed before the next chunk starts, so peak RAM
is bounded by one chunk's candidates, not the whole dataset. Recall
accounting is done incrementally in the same pass (no second full scan).

Set `RUN_LIMIT_S1` to a small number for a smoke test before committing to
the full 2.2M-row run.


In [29]:
# ============================================================
# 11. FINAL OPTIMIZED PIPELINE
#     Blocking + Top-K + Token Rescue + 20 Features
# ============================================================

def run_full_pipeline(
    s1_path,
    s2_index,
    s3_index,
    gt_path=None,
    chunk_size=None,
    run_limit_s1=None,
    tag="run",
    prefilter_k=30,
    rescue_name_threshold=0.67,
    rescue_address_threshold=0.50,
):
    """
    Final optimized candidate pipeline.

    FLOW:

        S1
         ↓
        Existing blocking
         ↓
        ~100 candidates
         ↓
        Top-K by blocking strength
         +
        Token-overlap rescue
         ↓
        ~33 candidates
         ↓
        Existing 20 expensive features
         ↓
        Parquet shards

    The rescue stage exists because a genuine entity may have
    strong name/address similarity but weak blocking evidence.

    Example discovered during testing:

        S1:
        White Marketing Private Limited

        S3:
        Private White Marketing Ltd

    This candidate ranked #98 by blocking strength but was
    recovered by token overlap.
    """

    chunk_size = chunk_size or CONFIG["s1_chunk_size"]

    freq_caps = CONFIG["freq_caps"]

    indexes_by_source = {
        "S2": s2_index,
        "S3": s3_index,
    }

    ground_truth = (
        load_ground_truth(gt_path)
        if gt_path
        else None
    )

    has_labels = ground_truth is not None

    # --------------------------------------------------------
    # Recall statistics
    # --------------------------------------------------------

    recall_stats = {
        source: {
            "actual": 0,
            "found": 0,
        }
        for source in indexes_by_source
    }

    combined_recall = {
        "actual": 0,
        "found": 0,
    }

    # --------------------------------------------------------
    # Counters
    # --------------------------------------------------------

    processed = 0

    total_pairs = 0

    total_base_pairs = 0

    total_rescue_pairs = 0

    total_prefilter_pairs = 0

    positive = 0

    negative = 0

    shard_idx = 0

    t0 = time.time()

    # ========================================================
    # TOKEN HELPERS
    # ========================================================

    def normalize_tokens(text):

        text = str(text or "").lower()

        text = re.sub(
            r"[^a-z0-9]+",
            " ",
            text,
        )

        return set(
            token
            for token in text.split()
            if len(token) >= 2
        )

    def overlap_score(tokens_a, tokens_b):

        if not tokens_a or not tokens_b:
            return 0.0

        return (
            len(tokens_a & tokens_b)
            /
            min(
                len(tokens_a),
                len(tokens_b),
            )
        )

    # ========================================================
    # MAIN LOOP
    # ========================================================

    for chunk in iter_s1_chunks(
        s1_path,
        chunk_size,
        limit=run_limit_s1,
    ):

        shard_rows = []

        for s1_row in chunk:

            s1_id = s1_row.get(
                "entity_id",
                "",
            )

            # =================================================
            # STEP 1
            # Existing blocking
            # =================================================

            pairs = list(
                generate_candidate_pairs_for_row(
                    s1_row,
                    indexes_by_source,
                    freq_caps,
                )
            )

            total_pairs += len(pairs)

            # =================================================
            # STEP 2
            # Ground-truth blocking recall
            # =================================================

            if has_labels:

                gt_entry = ground_truth.get(
                    s1_id,
                    {
                        "S2": set(),
                        "S3": set(),
                        "all": set(),
                    },
                )

                row_cand_ids = {
                    "S2": set(),
                    "S3": set(),
                }

                for pair in pairs:

                    source = pair["source"]

                    candidate_id = pair[
                        "candidate_entity_id"
                    ]

                    row_cand_ids[source].add(
                        candidate_id
                    )

                for source in indexes_by_source:

                    true_matches = gt_entry.get(
                        source,
                        set(),
                    )

                    recall_stats[source]["actual"] += len(
                        true_matches
                    )

                    recall_stats[source]["found"] += len(
                        true_matches
                        &
                        row_cand_ids[source]
                    )

                combined_recall["actual"] += len(
                    gt_entry.get("all", set())
                )

                combined_recall["found"] += len(
                    gt_entry.get("all", set())
                    &
                    (
                        row_cand_ids["S2"]
                        |
                        row_cand_ids["S3"]
                    )
                )

            # =================================================
            # STEP 3
            # Base ranking
            # =================================================

            pairs.sort(
                key=lambda p: (
                    -p.get(
                        "blocking_num_keys",
                        0,
                    ),

                    p.get(
                        "blocking_min_freq",
                        float("inf"),
                    ),

                    p.get(
                        "candidate_entity_id",
                        "",
                    ),
                )
            )

            base_pairs = pairs[:prefilter_k]

            total_base_pairs += len(
                base_pairs
            )

            # -------------------------------------------------
            # Selected candidate IDs
            # -------------------------------------------------

            selected_ids = {
                p["candidate_entity_id"]
                for p in base_pairs
            }

            # =================================================
            # STEP 4
            # Token rescue
            # =================================================

            s1_name = s1_row.get(
                "business_name",
                "",
            )

            s1_addr = s1_row.get(
                "business_address",
                "",
            )

            s1_country = s1_row.get(
                "country",
                "",
            )

            s1_name_tokens = normalize_tokens(
                s1_name
            )

            s1_addr_tokens = normalize_tokens(
                s1_addr
            )

            rescue_pairs = []

            # Only examine candidates that were NOT
            # already retained by the base top-K.
            #
            # This avoids duplicate work.

            for pair in pairs[prefilter_k:]:

                candidate_id = pair[
                    "candidate_entity_id"
                ]

                if candidate_id in selected_ids:
                    continue

                source = pair["source"]

                _, suffix = split_entity_id(
                    candidate_id
                )

                attrs = indexes_by_source[
                    source
                ].get_attrs(
                    suffix
                )

                if not attrs:
                    continue

                cand_name = attrs[0] or ""

                cand_addr = attrs[1] or ""

                # --------------------------------------------
                # Name overlap
                # --------------------------------------------

                cand_name_tokens = normalize_tokens(
                    cand_name
                )

                name_overlap = overlap_score(
                    s1_name_tokens,
                    cand_name_tokens,
                )

                # --------------------------------------------
                # Address overlap
                # --------------------------------------------

                cand_addr_tokens = normalize_tokens(
                    cand_addr
                )

                address_overlap = overlap_score(
                    s1_addr_tokens,
                    cand_addr_tokens,
                )

                # --------------------------------------------
                # Rescue rule
                # --------------------------------------------

                rescue = (
                    name_overlap
                    >= rescue_name_threshold
                    or
                    address_overlap
                    >= rescue_address_threshold
                )

                if rescue:

                    selected_ids.add(
                        candidate_id
                    )

                    rescue_pairs.append(
                        pair
                    )

            total_rescue_pairs += len(
                rescue_pairs
            )

            # =================================================
            # STEP 5
            # Final candidate set
            # =================================================

            final_pairs = (
                base_pairs
                +
                rescue_pairs
            )

            total_prefilter_pairs += len(
                final_pairs
            )

            # =================================================
            # STEP 6
            # Existing expensive 20 features
            # =================================================

            for pair in final_pairs:

                source = pair["source"]

                candidate_id = pair[
                    "candidate_entity_id"
                ]

                _, suffix = split_entity_id(
                    candidate_id
                )

                attrs = indexes_by_source[
                    source
                ].get_attrs(
                    suffix
                )

                if attrs:

                    (
                        cand_name,
                        cand_addr,
                        cand_country,
                    ) = attrs

                else:

                    cand_name = ""

                    cand_addr = ""

                    cand_country = ""

                feats = calculate_features(
                    s1_name,
                    s1_addr,
                    s1_country,
                    cand_name,
                    cand_addr,
                    cand_country,
                )

                out_row = {
                    **pair,
                    **feats,
                }

                # ------------------------------------------------
                # Labels
                # ------------------------------------------------

                if has_labels:

                    label = label_for(
                        s1_id,
                        candidate_id,
                        ground_truth,
                    )

                    out_row["label"] = label

                    if label:

                        positive += 1

                    else:

                        negative += 1

                shard_rows.append(
                    out_row
                )

            processed += 1

        # =====================================================
        # STEP 7
        # Write shard
        # =====================================================

        if shard_rows:

            shard_path = (
                SHARD_DIR
                /
                f"{tag}_shard_{shard_idx:05d}.parquet"
            )

            pd.DataFrame(
                shard_rows
            ).to_parquet(
                shard_path,
                index=False,
            )

        shard_idx += 1

        del shard_rows

        gc.collect()

        # =====================================================
        # Progress report
        # =====================================================

        elapsed = time.time() - t0

        resource_report(
            f"{tag} chunk {shard_idx}",
            s1_processed=processed,
            candidate_pairs=total_pairs,
            elapsed_s=round(
                elapsed,
                1,
            ),
        )

        print(
            f"  Base top-{prefilter_k}: "
            f"{total_base_pairs:,}"
        )

        print(
            f"  Rescue pairs: "
            f"{total_rescue_pairs:,}"
        )

        print(
            f"  Final feature pairs: "
            f"{total_prefilter_pairs:,}"
        )

        if total_pairs:

            print(
                f"  Final retention: "
                f"{100 * total_prefilter_pairs / total_pairs:.2f}%"
            )

    # ========================================================
    # FINAL SUMMARY
    # ========================================================

    summary = {

        "tag": tag,

        "s1_processed": processed,

        "candidate_pairs": total_pairs,

        "base_pairs": total_base_pairs,

        "rescue_pairs": total_rescue_pairs,

        "prefilter_pairs": total_prefilter_pairs,

        "avg_candidates_per_s1": (
            total_pairs / processed
            if processed
            else 0
        ),

        "avg_base_pairs_per_s1": (
            total_base_pairs / processed
            if processed
            else 0
        ),

        "avg_rescue_pairs_per_s1": (
            total_rescue_pairs / processed
            if processed
            else 0
        ),

        "avg_prefilter_pairs_per_s1": (
            total_prefilter_pairs / processed
            if processed
            else 0
        ),

        "prefilter_k": prefilter_k,

        "rescue_name_threshold":
            rescue_name_threshold,

        "rescue_address_threshold":
            rescue_address_threshold,

        "elapsed_s":
            time.time() - t0,

        "shards":
            shard_idx,
    }

    # ========================================================
    # LABEL / RECALL
    # ========================================================

    if has_labels:

        summary["positive_labels"] = positive

        summary["negative_labels"] = negative

        summary["recall_by_source"] = {
            source: (
                stats["found"]
                /
                stats["actual"]
                if stats["actual"]
                else None
            )
            for source, stats
            in recall_stats.items()
        }

        summary["recall_combined"] = (
            combined_recall["found"]
            /
            combined_recall["actual"]
            if combined_recall["actual"]
            else None
        )

    return summary

## 12. Run it (smoke test first, then full scale)

**Run the smoke test cell first** (`RUN_LIMIT_S1` set small) to confirm the
pipeline executes correctly in this Kaggle environment before committing to
the multi-hour full run. Only then set `RUN_LIMIT_S1 = None` for the full
2,206,821-row TRAIN pass.


In [30]:
SMOKE_TEST_S1_LIMIT = 1000

smoke_summary = run_full_pipeline(
    S1_PATH,
    s2_index,
    s3_index,
    gt_path=GT_PATH,
    run_limit_s1=SMOKE_TEST_S1_LIMIT,
    tag="final_1000",
    prefilter_k=30,
    rescue_name_threshold=0.67,
    rescue_address_threshold=0.50,
)

print(smoke_summary)

STAGE: final_1000 chunk 1
  s1_processed: 1000
  candidate_pairs: 100248
  elapsed_s: 28.6
  RAM  -> process=6.53GB  system_used=7.36GB  system_available=25.82GB
  DISK -> used=8.68GB  free=12.26GB
  Base top-30: 27,339
  Rescue pairs: 5,586
  Final feature pairs: 32,925
  Final retention: 32.84%
{'tag': 'final_1000', 's1_processed': 1000, 'candidate_pairs': 100248, 'base_pairs': 27339, 'rescue_pairs': 5586, 'prefilter_pairs': 32925, 'avg_candidates_per_s1': 100.248, 'avg_base_pairs_per_s1': 27.339, 'avg_rescue_pairs_per_s1': 5.586, 'avg_prefilter_pairs_per_s1': 32.925, 'prefilter_k': 30, 'rescue_name_threshold': 0.67, 'rescue_address_threshold': 0.5, 'elapsed_s': 28.576266050338745, 'shards': 1, 'positive_labels': 11, 'negative_labels': 32914, 'recall_by_source': {'S2': 0.0017584994138335288, 'S3': 0.004417448923246825}, 'recall_combined': 0.0031276656241114587}


In [31]:
import shutil

total, used, free = shutil.disk_usage("/kaggle/working")

print(f"Total disk: {total / 1024**3:.2f} GB")
print(f"Used disk:  {used / 1024**3:.2f} GB")
print(f"Free disk:  {free / 1024**3:.2f} GB")

Total disk: 19.52 GB
Used disk:  8.09 GB
Free disk:  11.41 GB


In [32]:
import inspect

print("build_index signature:")
print(inspect.signature(build_index))

print("\nSqliteIndex methods:")
print([
    x for x in dir(SqliteIndex)
    if not x.startswith("_")
])

print("\nMemoryIndex methods:")
print([
    x for x in dir(MemoryIndex)
    if not x.startswith("_")
])

build_index signature:
(path, source_prefix, backend='sqlite', db_path=None, limit=None)

SqliteIndex methods:
['add_record', 'bucket', 'finalize', 'get_attrs', 'num_keys']

MemoryIndex methods:
['add_record', 'bucket', 'get_attrs', 'num_keys']


In [24]:
# ============================================================
# PREFILTER RECALL DIAGNOSTIC
# ============================================================

DIAG_LIMIT = 1000
PREFILTER_K = 30

gt = load_ground_truth(GT_PATH)

total_candidates = 0
total_true_in_candidates = 0
total_true_after_filter = 0

rows_with_true_candidate = 0
rows_with_true_after_filter = 0

processed = 0

t0 = time.time()

for chunk in iter_s1_chunks(
    S1_PATH,
    CONFIG["s1_chunk_size"],
    limit=DIAG_LIMIT,
):

    for s1_row in chunk:

        s1_id = s1_row.get(
            "entity_id",
            "",
        )

        pairs = list(
            generate_candidate_pairs_for_row(
                s1_row,
                {
                    "S2": s2_index,
                    "S3": s3_index,
                },
                CONFIG["freq_caps"],
            )
        )

        total_candidates += len(pairs)

        # ----------------------------------------------------
        # True matches according to ground truth
        # ----------------------------------------------------

        gt_entry = gt.get(
            s1_id,
            {
                "S2": set(),
                "S3": set(),
                "all": set(),
            },
        )

        true_ids = set(
            gt_entry.get("all", set())
        )

        candidate_ids = {
            p["candidate_entity_id"]
            for p in pairs
        }

        true_in_candidates = (
            true_ids & candidate_ids
        )

        if true_in_candidates:
            rows_with_true_candidate += 1

        total_true_in_candidates += len(
            true_in_candidates
        )

        # ----------------------------------------------------
        # Apply SAME top-K ranking as fast pipeline
        # ----------------------------------------------------

        if len(pairs) > PREFILTER_K:

            pairs.sort(
                key=lambda p: (
                    -p.get(
                        "blocking_num_keys",
                        0,
                    ),

                    p.get(
                        "blocking_min_freq",
                        float("inf"),
                    ),

                    p.get(
                        "candidate_entity_id",
                        "",
                    ),
                )
            )

            pairs = pairs[:PREFILTER_K]

        filtered_ids = {
            p["candidate_entity_id"]
            for p in pairs
        }

        true_after_filter = (
            true_ids & filtered_ids
        )

        if true_after_filter:
            rows_with_true_after_filter += 1

        total_true_after_filter += len(
            true_after_filter
        )

        processed += 1

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

blocking_recall = (
    total_true_in_candidates
    / len([
        x
        for s1_id in gt
        if (
            s1_id
            and gt[s1_id].get("all", set())
        )
    ])
    if False
    else None
)

prefilter_recall = (
    total_true_after_filter
    / total_true_in_candidates
    if total_true_in_candidates
    else None
)

print("=" * 70)
print("PREFILTER RECALL DIAGNOSTIC")
print("=" * 70)

print(f"S1 rows tested:                 {processed:,}")
print(f"Candidate pairs:                {total_candidates:,}")
print(
    f"True matches found in candidates: "
    f"{total_true_in_candidates:,}"
)
print(
    f"True matches after top-{PREFILTER_K}: "
    f"{total_true_after_filter:,}"
)

print()
print(
    f"Rows with true candidate:        "
    f"{rows_with_true_candidate:,}"
)

print(
    f"Rows with true after filter:     "
    f"{rows_with_true_after_filter:,}"
)

print()

if total_true_in_candidates:
    print(
        f"Prefilter recall: "
        f"{prefilter_recall * 100:.2f}%"
    )
else:
    print(
        "Prefilter recall: N/A "
        "(no ground-truth matches were present "
        "in the current 20k/20k test indexes)"
    )

print()
print(
    f"Elapsed: {time.time() - t0:.1f} sec"
)
print("=" * 70)

PREFILTER RECALL DIAGNOSTIC
S1 rows tested:                 1,000
Candidate pairs:                100,248
True matches found in candidates: 11
True matches after top-30: 10

Rows with true candidate:        11
Rows with true after filter:     10

Prefilter recall: 90.91%

Elapsed: 0.6 sec


In [25]:
# ============================================================
# TEST PREFILTER K VALUES
# ============================================================

K_VALUES = [30, 40, 50, 60]

gt = load_ground_truth(GT_PATH)

results = []

for K in K_VALUES:

    total_candidates = 0
    total_true_in_candidates = 0
    total_true_after_filter = 0

    processed = 0

    for chunk in iter_s1_chunks(
        S1_PATH,
        CONFIG["s1_chunk_size"],
        limit=1000,
    ):

        for s1_row in chunk:

            s1_id = s1_row.get(
                "entity_id",
                "",
            )

            pairs = list(
                generate_candidate_pairs_for_row(
                    s1_row,
                    {
                        "S2": s2_index,
                        "S3": s3_index,
                    },
                    CONFIG["freq_caps"],
                )
            )

            total_candidates += len(pairs)

            gt_entry = gt.get(
                s1_id,
                {
                    "S2": set(),
                    "S3": set(),
                    "all": set(),
                },
            )

            true_ids = set(
                gt_entry.get("all", set())
            )

            candidate_ids = {
                p["candidate_entity_id"]
                for p in pairs
            }

            true_in_candidates = (
                true_ids & candidate_ids
            )

            total_true_in_candidates += len(
                true_in_candidates
            )

            # Same ranking used by fast pipeline
            if len(pairs) > K:

                pairs.sort(
                    key=lambda p: (
                        -p.get(
                            "blocking_num_keys",
                            0,
                        ),

                        p.get(
                            "blocking_min_freq",
                            float("inf"),
                        ),

                        p.get(
                            "candidate_entity_id",
                            "",
                        ),
                    )
                )

                pairs = pairs[:K]

            filtered_ids = {
                p["candidate_entity_id"]
                for p in pairs
            }

            true_after_filter = (
                true_ids & filtered_ids
            )

            total_true_after_filter += len(
                true_after_filter
            )

            processed += 1

    recall = (
        total_true_after_filter
        / total_true_in_candidates
        if total_true_in_candidates
        else None
    )

    results.append({
        "K": K,
        "candidates": total_candidates,
        "true_in_candidates": total_true_in_candidates,
        "true_after_filter": total_true_after_filter,
        "recall": recall,
    })


print("=" * 75)
print("PREFILTER K COMPARISON")
print("=" * 75)

for r in results:

    print(
        f"K={r['K']:>2} | "
        f"True: {r['true_after_filter']:>2}/"
        f"{r['true_in_candidates']:<2} | "
        f"Recall: {r['recall'] * 100:6.2f}%"
    )

print("=" * 75)

PREFILTER K COMPARISON
K=30 | True: 10/11 | Recall:  90.91%
K=40 | True: 10/11 | Recall:  90.91%
K=50 | True: 10/11 | Recall:  90.91%
K=60 | True: 10/11 | Recall:  90.91%


In [26]:
# ============================================================
# DIAGNOSE THE TRUE MATCH LOST BY PREFILTER
# ============================================================

DIAG_LIMIT = 1000

gt = load_ground_truth(GT_PATH)

found_lost = False

for chunk in iter_s1_chunks(
    S1_PATH,
    CONFIG["s1_chunk_size"],
    limit=DIAG_LIMIT,
):

    for s1_row in chunk:

        s1_id = s1_row.get(
            "entity_id",
            "",
        )

        gt_entry = gt.get(
            s1_id,
            {
                "S2": set(),
                "S3": set(),
                "all": set(),
            },
        )

        true_ids = set(
            gt_entry.get("all", set())
        )

        if not true_ids:
            continue

        pairs = list(
            generate_candidate_pairs_for_row(
                s1_row,
                {
                    "S2": s2_index,
                    "S3": s3_index,
                },
                CONFIG["freq_caps"],
            )
        )

        candidate_ids = {
            p["candidate_entity_id"]
            for p in pairs
        }

        true_in_candidates = (
            true_ids & candidate_ids
        )

        if not true_in_candidates:
            continue

        # Same ranking as our current fast pipeline
        ranked_pairs = sorted(
            pairs,
            key=lambda p: (
                -p.get(
                    "blocking_num_keys",
                    0,
                ),

                p.get(
                    "blocking_min_freq",
                    float("inf"),
                ),

                p.get(
                    "candidate_entity_id",
                    "",
                ),
            )
        )

        ranked_ids = [
            p["candidate_entity_id"]
            for p in ranked_pairs
        ]

        # Find true matches that blocking found
        # but top-60 did not retain.
        lost = [
            cid
            for cid in true_in_candidates
            if cid not in ranked_ids[:60]
        ]

        if lost:

            print("=" * 75)
            print("LOST TRUE MATCH DIAGNOSTIC")
            print("=" * 75)

            print(f"S1 entity_id: {s1_id}")

            print()
            print("S1 business name:")
            print(s1_row.get("business_name", ""))

            print()
            print("S1 address:")
            print(s1_row.get("business_address", ""))

            print()
            print("S1 country:")
            print(s1_row.get("country", ""))

            print()
            print("TRUE MATCH(ES) LOST:")
            print(lost)

            print()
            print("DETAILS OF LOST MATCH:")
            print("-" * 75)

            for lost_id in lost:

                matching_pair = next(
                    (
                        p
                        for p in ranked_pairs
                        if p["candidate_entity_id"]
                        == lost_id
                    ),
                    None,
                )

                if matching_pair:

                    rank = (
                        ranked_ids.index(lost_id)
                        + 1
                    )

                    print(
                        f"Candidate: {lost_id}"
                    )

                    print(
                        f"Rank: {rank}"
                    )

                    print(
                        "blocking_num_keys:",
                        matching_pair.get(
                            "blocking_num_keys"
                        ),
                    )

                    print(
                        "blocking_min_freq:",
                        matching_pair.get(
                            "blocking_min_freq"
                        ),
                    )

                    print()

                    source = matching_pair["source"]

                    _, suffix = split_entity_id(
                        lost_id
                    )

                    attrs = indexes_by_source[
                        source
                    ].get_attrs(
                        suffix
                    )

                    print(
                        "Candidate attributes:"
                    )

                    print(
                        "  Name:",
                        attrs[0] if attrs else "",
                    )

                    print(
                        "  Address:",
                        attrs[1] if attrs else "",
                    )

                    print(
                        "  Country:",
                        attrs[2] if attrs else "",
                    )

                    print()

            print("=" * 75)

            found_lost = True
            break

    if found_lost:
        break

if not found_lost:

    print(
        "No lost true match found in the first "
        f"{DIAG_LIMIT} S1 rows."
    )

LOST TRUE MATCH DIAGNOSTIC
S1 entity_id: S1-38402575

S1 business name:
White Marketing Private Limited

S1 address:
S. No 142/4, Cts No 5102, Fl No 1001, Bld No C, 10Th Floor, Aishwaryam Ventures, Pune City, Pune, Maharashtra

S1 country:
India

TRUE MATCH(ES) LOST:
['S3-886031599']

DETAILS OF LOST MATCH:
---------------------------------------------------------------------------
Candidate: S3-886031599
Rank: 98
blocking_num_keys: 1
blocking_min_freq: 42

Candidate attributes:
  Name: Private White Marketing Ltd
  Address: Pune, Pune City, null, MH, H.no 505 S. No 142/4
  Country: India



In [27]:
# ============================================================
# TOKEN-OVERLAP RESCUE TEST
# ============================================================

import re

def normalize_tokens(text):
    text = str(text or "").lower()

    # Keep letters/numbers, turn everything else into spaces
    text = re.sub(r"[^a-z0-9]+", " ", text)

    return set(
        token
        for token in text.split()
        if len(token) >= 2
    )


def token_jaccard(a, b):
    A = normalize_tokens(a)
    B = normalize_tokens(b)

    if not A or not B:
        return 0.0

    return len(A & B) / len(A | B)


def token_overlap_score(a, b):
    """
    Recall-friendly overlap score.

    Uses the fraction of the smaller token set that
    is shared with the other string.
    """

    A = normalize_tokens(a)
    B = normalize_tokens(b)

    if not A or not B:
        return 0.0

    return len(A & B) / min(len(A), len(B))


# ------------------------------------------------------------
# Check the known lost match directly
# ------------------------------------------------------------

s1_name = "White Marketing Private Limited"
s3_name = "Private White Marketing Ltd"

s1_addr = (
    "S. No 142/4, Cts No 5102, Fl No 1001, "
    "Bld No C, 10Th Floor, Aishwaryam Ventures, "
    "Pune City, Pune, Maharashtra"
)

s3_addr = (
    "Pune, Pune City, null, MH, "
    "H.no 505 S. No 142/4"
)

print("=" * 70)
print("TOKEN RESCUE TEST")
print("=" * 70)

print()
print("NAME")
print("S1:", s1_name)
print("S3:", s3_name)

print(
    "Name Jaccard:",
    round(
        token_jaccard(
            s1_name,
            s3_name,
        ),
        4,
    ),
)

print(
    "Name overlap:",
    round(
        token_overlap_score(
            s1_name,
            s3_name,
        ),
        4,
    ),
)

print()
print("ADDRESS")

print(
    "Address Jaccard:",
    round(
        token_jaccard(
            s1_addr,
            s3_addr,
        ),
        4,
    ),
)

print(
    "Address overlap:",
    round(
        token_overlap_score(
            s1_addr,
            s3_addr,
        ),
        4,
    ),
)

print()
print("Shared name tokens:")
print(
    normalize_tokens(s1_name)
    &
    normalize_tokens(s3_name)
)

print()
print("Shared address tokens:")
print(
    normalize_tokens(s1_addr)
    &
    normalize_tokens(s3_addr)
)

print("=" * 70)

TOKEN RESCUE TEST

NAME
S1: White Marketing Private Limited
S3: Private White Marketing Ltd
Name Jaccard: 0.6
Name overlap: 0.75

ADDRESS
Address Jaccard: 0.2353
Address overlap: 0.5714

Shared name tokens:
{'marketing', 'private', 'white'}

Shared address tokens:
{'city', 'no', 'pune', '142'}


In [28]:
# ============================================================
# TOKEN-OVERLAP RESCUE RECALL TEST
# ============================================================

RESCUE_NAME_THRESHOLD = 0.67
RESCUE_ADDRESS_THRESHOLD = 0.50

DIAG_LIMIT = 1000
BASE_K = 30

gt = load_ground_truth(GT_PATH)

total_candidates = 0
total_base = 0
total_rescued = 0

total_true_in_candidates = 0
total_true_after_rescue = 0

rows_with_true_candidate = 0
rows_with_true_after_rescue = 0

processed = 0


for chunk in iter_s1_chunks(
    S1_PATH,
    CONFIG["s1_chunk_size"],
    limit=DIAG_LIMIT,
):

    for s1_row in chunk:

        s1_id = s1_row.get(
            "entity_id",
            "",
        )

        pairs = list(
            generate_candidate_pairs_for_row(
                s1_row,
                {
                    "S2": s2_index,
                    "S3": s3_index,
                },
                CONFIG["freq_caps"],
            )
        )

        total_candidates += len(pairs)

        # ----------------------------------------------------
        # Ground truth
        # ----------------------------------------------------

        gt_entry = gt.get(
            s1_id,
            {
                "S2": set(),
                "S3": set(),
                "all": set(),
            },
        )

        true_ids = set(
            gt_entry.get("all", set())
        )

        candidate_ids = {
            p["candidate_entity_id"]
            for p in pairs
        }

        true_in_candidates = (
            true_ids & candidate_ids
        )

        total_true_in_candidates += len(
            true_in_candidates
        )

        if true_in_candidates:
            rows_with_true_candidate += 1

        # ----------------------------------------------------
        # Base ranking
        # ----------------------------------------------------

        ranked = sorted(
            pairs,
            key=lambda p: (
                -p.get(
                    "blocking_num_keys",
                    0,
                ),

                p.get(
                    "blocking_min_freq",
                    float("inf"),
                ),

                p.get(
                    "candidate_entity_id",
                    "",
                ),
            )
        )

        base_pairs = ranked[:BASE_K]

        selected_ids = {
            p["candidate_entity_id"]
            for p in base_pairs
        }

        total_base += len(base_pairs)

        # ----------------------------------------------------
        # Rescue candidates outside top-K
        # ----------------------------------------------------

        s1_name = s1_row.get(
            "business_name",
            "",
        )

        s1_addr = s1_row.get(
            "business_address",
            "",
        )

        s1_name_tokens = normalize_tokens(
            s1_name
        )

        s1_addr_tokens = normalize_tokens(
            s1_addr
        )

        for pair in ranked[BASE_K:]:

            candidate_id = pair[
                "candidate_entity_id"
            ]

            source = pair["source"]

            _, suffix = split_entity_id(
                candidate_id
            )

            attrs = indexes_by_source[
                source
            ].get_attrs(
                suffix
            )

            if not attrs:
                continue

            cand_name = attrs[0] or ""
            cand_addr = attrs[1] or ""

            cand_name_tokens = normalize_tokens(
                cand_name
            )

            cand_addr_tokens = normalize_tokens(
                cand_addr
            )

            # ----------------------------------------------
            # Name overlap
            # ----------------------------------------------

            if (
                s1_name_tokens
                and cand_name_tokens
            ):

                name_overlap = (
                    len(
                        s1_name_tokens
                        &
                        cand_name_tokens
                    )
                    /
                    min(
                        len(s1_name_tokens),
                        len(cand_name_tokens),
                    )
                )

            else:

                name_overlap = 0.0

            # ----------------------------------------------
            # Address overlap
            # ----------------------------------------------

            if (
                s1_addr_tokens
                and cand_addr_tokens
            ):

                address_overlap = (
                    len(
                        s1_addr_tokens
                        &
                        cand_addr_tokens
                    )
                    /
                    min(
                        len(s1_addr_tokens),
                        len(cand_addr_tokens),
                    )
                )

            else:

                address_overlap = 0.0

            # ----------------------------------------------
            # Rescue condition
            # ----------------------------------------------

            rescue = (
                name_overlap
                >= RESCUE_NAME_THRESHOLD
                or
                address_overlap
                >= RESCUE_ADDRESS_THRESHOLD
            )

            if rescue:

                selected_ids.add(
                    candidate_id
                )

                total_rescued += 1

        total_rescued_here = (
            len(selected_ids) - len(base_pairs)
        )

        # ----------------------------------------------------
        # True matches after rescue
        # ----------------------------------------------------

        true_after = (
            true_ids & selected_ids
        )

        total_true_after_rescue += len(
            true_after
        )

        if true_after:
            rows_with_true_after_rescue += 1

        processed += 1


# ============================================================
# RESULTS
# ============================================================

print("=" * 75)
print("TOKEN-OVERLAP RESCUE RECALL TEST")
print("=" * 75)

print(
    f"S1 rows tested:                 "
    f"{processed:,}"
)

print(
    f"Candidate pairs:                "
    f"{total_candidates:,}"
)

print(
    f"Base top-{BASE_K} pairs:             "
    f"{total_base:,}"
)

print(
    f"Rescue pairs added:             "
    f"{total_rescued:,}"
)

print()

print(
    f"True matches in candidates:     "
    f"{total_true_in_candidates:,}"
)

print(
    f"True matches after rescue:      "
    f"{total_true_after_rescue:,}"
)

print()

print(
    f"Rows with true candidate:       "
    f"{rows_with_true_candidate:,}"
)

print(
    f"Rows with true after rescue:    "
    f"{rows_with_true_after_rescue:,}"
)

if total_true_in_candidates:

    recall = (
        total_true_after_rescue
        /
        total_true_in_candidates
    )

    print()
    print(
        f"Rescue recall:                  "
        f"{recall * 100:.2f}%"
    )

print()
print(
    f"Final retained pairs:           "
    f"{total_base + total_rescued:,}"
)

print(
    f"Average retained / S1:          "
    f"{(total_base + total_rescued) / processed:.2f}"
)

print("=" * 75)

TOKEN-OVERLAP RESCUE RECALL TEST
S1 rows tested:                 1,000
Candidate pairs:                100,248
Base top-30 pairs:             27,339
Rescue pairs added:             5,586

True matches in candidates:     11
True matches after rescue:      11

Rows with true candidate:       11
Rows with true after rescue:    11

Rescue recall:                  100.00%

Final retained pairs:           32,925
Average retained / S1:          32.92


In [ ]:
from pathlib import Path
import shutil
import gc

# Remove the huge SQLite indexes
for p in [
    Path("/kaggle/working/index"),
    Path("/kaggle/working/feature_shards"),
]:
    if p.exists():
        print("Deleting:", p)
        shutil.rmtree(p, ignore_errors=True)

gc.collect()

# Check remaining disk
total, used, free = shutil.disk_usage("/kaggle/working")

print(f"Disk used : {used/1e9:.2f} GB")
print(f"Disk free : {free/1e9:.2f} GB")

In [ ]:
# ============================================================
# SAFE SMOKE TEST — DO NOT BUILD FULL 5M+ RECORD INDEXES
# ============================================================

import time
import gc
import json
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

SMOKE_S1 = 100          # Start VERY small
SMOKE_S2 = 10_000       # Only read a small S2 sample
SMOKE_S3 = 10_000       # Only read a small S3 sample

print("=" * 70)
print("SAFE SMOKE TEST")
print("=" * 70)

# ------------------------------------------------------------
# 1. Check disk FIRST
# ------------------------------------------------------------

total, used, free = shutil.disk_usage("/kaggle/working")

print(f"Disk total : {total/1e9:.2f} GB")
print(f"Disk used  : {used/1e9:.2f} GB")
print(f"Disk free  : {free/1e9:.2f} GB")

if free < 2e9:
    raise RuntimeError(
        "STOP: Less than 2 GB disk space remains. "
        "Delete old index/shard files before continuing."
    )

# ------------------------------------------------------------
# 2. Load ONLY tiny samples
# ------------------------------------------------------------

t0 = time.time()

print("\nLoading S1 sample...")
s1_sample = pd.read_csv(
    S1_PATH,
    sep="\t",
    nrows=SMOKE_S1,
)

print("S1:", s1_sample.shape)

print("\nLoading S2 sample...")
s2_sample = pd.read_csv(
    S2_PATH,
    sep="\t",
    nrows=SMOKE_S2,
)

print("S2:", s2_sample.shape)

print("\nLoading S3 sample...")
s3_sample = pd.read_csv(
    S3_PATH,
    sep="\t",
    nrows=SMOKE_S3,
)

print("S3:", s3_sample.shape)

print(f"\nLoading time: {time.time() - t0:.2f}s")

# ------------------------------------------------------------
# 3. Check columns
# ------------------------------------------------------------

print("\nS1 columns:")
print(s1_sample.columns.tolist())

print("\nS2 columns:")
print(s2_sample.columns.tolist())

print("\nS3 columns:")
print(s3_sample.columns.tolist())

# ------------------------------------------------------------
# 4. Build SMALL in-memory indexes
# ------------------------------------------------------------

print("\nBuilding SMALL test indexes...")

small_index_s2 = build_index(
    S2_PATH,
    "S2",
    backend="memory",
    limit=SMOKE_S2,
)

small_index_s3 = build_index(
    S3_PATH,
    "S3",
    backend="memory",
    limit=SMOKE_S3,
)

indexes_small = {
    "S2": small_index_s2,
    "S3": small_index_s3,
}

print("\nSmall indexes ready.")

# ------------------------------------------------------------
# 5. Generate candidates for ONLY 100 S1 rows
# ------------------------------------------------------------

print("\nTesting candidate generation...")

t0 = time.time()

candidate_count = 0
tested_rows = 0

for _, row in s1_sample.iterrows():

    s1_row = row.to_dict()

    pairs = generate_candidate_pairs_for_row(
        s1_row,
        indexes_small,
        CONFIG["freq_caps"],
    )

    candidate_count += len(pairs)
    tested_rows += 1

    if tested_rows % 10 == 0:
        print(
            f"Processed {tested_rows}/{SMOKE_S1} S1 rows | "
            f"Candidates: {candidate_count}"
        )

elapsed = time.time() - t0

print("\n" + "=" * 70)
print("SMOKE TEST COMPLETE")
print("=" * 70)

print(f"S1 rows tested          : {tested_rows}")
print(f"Candidate pairs         : {candidate_count}")
print(
    f"Candidates per S1       : "
    f"{candidate_count / max(tested_rows, 1):.2f}"
)
print(f"Candidate generation    : {elapsed:.2f} sec")

# ------------------------------------------------------------
# 6. Resource report
# ------------------------------------------------------------

resource_report(
    "safe smoke test",
    s1_rows=tested_rows,
    candidate_pairs=candidate_count,
    elapsed_s=round(elapsed, 2),
)

# ------------------------------------------------------------
# 7. Cleanup
# ------------------------------------------------------------

del s1_sample
del s2_sample
del s3_sample
del small_index_s2
del small_index_s3
del indexes_small

gc.collect()

print("\nMemory cleaned.")

In [ ]:
# ---- FULL TRAIN RUN ----
# Uncomment to run against the entire 2,206,821-row S1 file.
# This is the expensive, multi-hour step -- only run it after the smoke
# test above looks correct (recall close to the 97.96% reference, no errors).

# RUN_LIMIT_S1 = None  # None = every S1 row
# full_summary = run_full_pipeline(
#     S1_PATH, s2_index, s3_index, gt_path=GT_PATH,
#     run_limit_s1=RUN_LIMIT_S1, tag="full_train",
# )
# print(json.dumps(full_summary, indent=2, default=str))


## 13. Resource + recall report

Summarizes candidates, recall, runtime, RAM and disk for whichever run
(`smoke_summary` and/or `full_summary`) has been executed above -- satisfies
Part 20's per-stage reporting requirement.


In [ ]:
def print_stage_report(summary):
    print("=" * 70)
    print(f"Stage: {summary['tag']}")
    print(f"S1 rows: {summary['s1_processed']:,}")
    print(f"Candidates: {summary['candidate_pairs']:,}")
    print(f"Candidates/S1: {summary['avg_candidates_per_s1']:.2f}")
    if "recall_by_source" in summary:
        for source, r in summary["recall_by_source"].items():
            print(f"{source} recall: {r*100:.2f}%" if r is not None else f"{source} recall: n/a")
        c = summary.get("recall_combined")
        print(f"Combined recall: {c*100:.2f}%" if c is not None else "Combined recall: n/a")
    print(f"Runtime: {summary['elapsed_s']:.1f}s")
    print(f"Shards written: {summary['shards']}")
    print("=" * 70)


print_stage_report(smoke_summary)
resource_report("after smoke test")


## 14. Train the matching model

Loads the Parquet shards written by Section 11/12 (never all candidates as
one in-memory DataFrame if it doesn't fit -- `pd.read_parquet` per shard,
concatenated only if it comfortably fits, otherwise switch to
`lgb.Dataset`'s native chunked/streaming construction). Splits by
**S1 entity** (not by row) to avoid leaking the same S1's candidates across
train/validation.


In [ ]:
def load_shards(tag):
    paths = sorted(SHARD_DIR.glob(f"{tag}_shard_*.parquet"))
    if not paths:
        raise FileNotFoundError(f"No shards found for tag={tag!r} in {SHARD_DIR}")
    return pd.concat((pd.read_parquet(p) for p in paths), ignore_index=True)


def train_val_split_by_s1(df, val_frac=0.2, seed=42):
    s1_ids = df["source1_entity_id"].unique()
    rng = random.Random(seed)
    s1_ids = list(s1_ids)
    rng.shuffle(s1_ids)
    n_val = int(len(s1_ids) * val_frac)
    val_ids = set(s1_ids[:n_val])
    val_mask = df["source1_entity_id"].isin(val_ids)
    return df[~val_mask].copy(), df[val_mask].copy()


def train_matcher(df):
    X = df[FEATURE_COLUMNS + ["blocking_num_keys", "blocking_min_freq"]]
    y = df["label"]

    train_df, val_df = train_val_split_by_s1(df)
    X_train = train_df[X.columns]
    y_train = train_df["label"]
    X_val = val_df[X.columns]
    y_val = val_df["label"]

    train_set = lgb.Dataset(X_train, label=y_train)
    val_set = lgb.Dataset(X_val, label=y_val, reference=train_set)

    params = {
        "objective": "binary",
        "metric": ["auc", "binary_logloss"],
        "boosting_type": "gbdt",
        "learning_rate": 0.05,
        "num_leaves": 63,
        "min_data_in_leaf": 50,
        "feature_fraction": 0.9,
        "bagging_fraction": 0.9,
        "bagging_freq": 1,
        "verbose": -1,
    }

    model = lgb.train(
        params, train_set, num_boost_round=500,
        valid_sets=[train_set, val_set], valid_names=["train", "val"],
        callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)],
    )

    val_df = val_df.copy()
    val_df["score"] = model.predict(X_val, num_iteration=model.best_iteration)
    return model, train_df, val_df


## 15. Validate: candidate recall vs. final match quality, and F0.5

Two genuinely different numbers, reported separately (never conflating
"98% candidate recall" with "98% matching accuracy"):

- **Candidate recall** (already computed in Section 11/13): did blocking
  include the true match at all?
- **Final match quality**: given a score threshold/top-k rule, does the
  *matcher* pick the right candidate(s)? Evaluated with the competition's
  actual metric, **macro-averaged F0.5 per S1 entity**, exactly as defined
  in the problem statement (singletons scored 1.0 for a correct empty
  prediction, 0.0 for any false-positive match).


In [ ]:
def f_beta(precision, recall, beta=0.5):
    if precision == 0 and recall == 0:
        return 0.0
    b2 = beta ** 2
    denom = (b2 * precision) + recall
    if denom == 0:
        return 0.0
    return (1 + b2) * precision * recall / denom


def evaluate_f05(val_df, ground_truth, threshold=0.5):
    """
    Macro-averages F0.5 per Source-1 entity, over every S1 entity that
    appears in val_df (predicted matches) UNIONED with every S1 entity in
    ground_truth that has at least one candidate in val_df -- i.e. S1
    entities the candidate stage produced zero candidates for still count,
    and score 0 unless they were true singletons (in which case they
    score 1.0 for predicting nothing).
    """
    preds = defaultdict(set)
    for row in val_df.itertuples():
        if row.score >= threshold:
            preds[row.source1_entity_id].add(row.candidate_entity_id)

    all_s1 = set(val_df["source1_entity_id"].unique())

    scores = []
    for s1_id in all_s1:
        predicted = preds.get(s1_id, set())
        true_matches = ground_truth.get(s1_id, {"all": set()})["all"]

        if not true_matches and not predicted:
            scores.append(1.0)
            continue
        if not predicted:
            scores.append(0.0)
            continue

        tp = len(predicted & true_matches)
        precision = tp / len(predicted) if predicted else 0.0
        recall = tp / len(true_matches) if true_matches else 0.0
        scores.append(f_beta(precision, recall, beta=0.5))

    return float(np.mean(scores)), len(scores)


# Example usage once smoke_summary shards + a trained model exist:
#
# df = load_shards("smoke")
# model, train_df, val_df = train_matcher(df)
# ground_truth = load_ground_truth(GT_PATH)
# f05, n = evaluate_f05(val_df, ground_truth, threshold=0.5)
# print(f"Macro F0.5 on validation ({n} S1 entities): {f05:.4f}")


## 16. Prepare for TEST inference (not run yet -- no test data provided)

This function is deliberately **not executed**. The competition's TEST
dataset has not been supplied yet. It's defined here, matching the exact
`matching_results.tsv` / `candidate_pairs.tsv` output format from the
problem statement, so it can be run as soon as test data is available,
without changing anything upstream.


In [ ]:
def run_test_inference(test_s1_path, test_s2_path, test_s3_path, model,
                        threshold=0.5, out_dir=None):
    """
    NOT YET RUN -- requires the competition TEST files, which have not
    been provided. Wires together the same index/candidate/feature code
    as TRAIN, scores every candidate with `model`, and writes both
    required output files in the exact format the problem statement
    specifies:

      matching_results.tsv : source1_entity_id, matched_entity_ids
      candidate_pairs.tsv  : source1_entity_id, candidate_entity_ids

    Every S1 test entity gets exactly one row in both files (empty
    matched_entity_ids for singletons), IDs restricted to those present
    in the test S2/S3 files, no duplicates within a list.
    """
    out_dir = Path(out_dir or OUT_DIR)

    test_s2_index = build_index(test_s2_path, "S2", backend=CONFIG["index_backend"],
                                 db_path=str(IDX_DIR / "test_s2_index.sqlite"))
    test_s3_index = build_index(test_s3_path, "S3", backend=CONFIG["index_backend"],
                                 db_path=str(IDX_DIR / "test_s3_index.sqlite"))
    indexes_by_source = {"S2": test_s2_index, "S3": test_s3_index}

    matching_rows = []
    candidate_rows = []

    for chunk in iter_s1_chunks(test_s1_path, CONFIG["s1_chunk_size"]):
        for s1_row in chunk:
            s1_id = s1_row.get("entity_id", "")
            pairs = generate_candidate_pairs_for_row(s1_row, indexes_by_source, CONFIG["freq_caps"])

            candidate_ids = []
            scored = []

            for pair in pairs:
                source = pair["source"]
                candidate_id = pair["candidate_entity_id"]
                candidate_ids.append(candidate_id)

                _, suffix = split_entity_id(candidate_id)
                attrs = indexes_by_source[source].get_attrs(suffix)
                cand_name, cand_addr, cand_country = attrs if attrs else ("", "", "")
                feats = calculate_features(
                    s1_row.get("business_name", ""), s1_row.get("business_address", ""),
                    s1_row.get("country", ""), cand_name, cand_addr, cand_country,
                )
                X = pd.DataFrame([{**feats,
                                    "blocking_num_keys": pair["blocking_num_keys"],
                                    "blocking_min_freq": pair["blocking_min_freq"]}])
                X = X[FEATURE_COLUMNS + ["blocking_num_keys", "blocking_min_freq"]]
                score = model.predict(X, num_iteration=model.best_iteration)[0]
                if score >= threshold:
                    scored.append(candidate_id)

            matching_rows.append({
                "source1_entity_id": s1_id,
                "matched_entity_ids": ",".join(dict.fromkeys(scored)),
            })
            candidate_rows.append({
                "source1_entity_id": s1_id,
                "candidate_entity_ids": ",".join(dict.fromkeys(candidate_ids)),
            })

    pd.DataFrame(matching_rows).to_csv(out_dir / "matching_results.tsv", sep="\t", index=False)
    pd.DataFrame(candidate_rows).to_csv(out_dir / "candidate_pairs.tsv", sep="\t", index=False)
    print(f"Wrote {out_dir/'matching_results.tsv'} and {out_dir/'candidate_pairs.tsv'}")


print("run_test_inference() is defined but NOT called -- no TEST dataset provided yet.")
